# 03 - RFM Segmentation

This notebook analyzes customer RFM (Recency, Frequency, Monetary) segmentation:
- Segment distributions
- Segment revenue contribution
- RFM heatmap
- References `outputs/metrics/rfm_summary.json` and `rfm_segment_details.json`

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Libraries loaded successfully.")

## 2. Load Data

In [ ]:
processed_path = Path('../data/processed')

customers = pd.read_csv(processed_path / 'customers_processed.csv')

rfm_cols = ['recency', 'frequency', 'monetary', 'rfm_score', 'rfm_segment']
available_rfm = [c for c in rfm_cols if c in customers.columns]

print(f'Customer data shape: {customers.shape}')
print(f'Available RFM columns: {available_rfm}')

## 3. Compute RFM if Missing

In [ ]:
if len(available_rfm) == 0:
    print("\nRFM columns not found. Computing from orders data...")
    orders = pd.read_csv(processed_path / 'orders_processed.csv')
    orders["order_date"] = pd.to_datetime(orders["order_date"])
    
    reference_date = orders['order_date'].max() + pd.Timedelta(days=1)
    
    rfm = orders.groupby('customer_id').agg(
        recency=('order_date', lambda x: (reference_date - x.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('revenue', 'sum')
    ).reset_index()
    
    rfm['r_score'] = pd.qcut(rfm['recency'], q=5, labels=[5,4,3,2,1], duplicates='drop')
    rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop')
    rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=5, labels=[1,2,3,4,5], duplicates='drop')
    rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)
    
    def assign_segment(row):
        r, f, m = int(row['r_score']), int(row['f_score']), int(row['m_score'])
        if r >= 4 and f >= 4 and m >= 4:
            return 'Champions'
        elif r >= 3 and f >= 3:
            return 'Loyal'
        elif r >= 4 and f <= 2:
            return 'New Customers'
        elif r <= 2 and f >= 3:
            return 'At Risk'
        elif r <= 2 and f <= 2:
            return 'Lost'
        else:
            return 'Potential'
    
    rfm['rfm_segment'] = rfm.apply(assign_segment, axis=1)
    customers = customers.merge(rfm, on='customer_id', how='left', suffixes=('', '_new'))
    
    if 'recency_new' in customers.columns:
        for col in ['recency', 'frequency', 'monetary', 'rfm_segment']:
            if col + '_new' in customers.columns:
                customers[col] = customers[col + '_new']
        customers.drop(columns=[c for c in customers.columns if c.endswith('_new')], inplace=True)
    
    available_rfm = ['recency', 'frequency', 'monetary', 'rfm_segment']
    print(f'RFM computed for {len(rfm)} customers.')
else:
    print("RFM columns already available.")

## 4. RFM Segment Distribution

In [ ]:
if 'rfm_segment' in customers.columns:
    segment_counts = customers['rfm_segment'].value_counts()
    segment_pct = (segment_counts / segment_counts.sum() * 100).round(2)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    segment_counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('RFM Segment Distribution')
    axes[0].set_ylabel('Number of Customers')
    axes[0].tick_params(axis='x', rotation=45)
    
    segment_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
    axes[1].set_title('RFM Segment Share')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    print('Segment Counts:')
    for seg, cnt in segment_counts.items():
        print(f'  {seg}: {cnt:,} ({segment_pct[seg]:.1f}%)')
else:
    print("rfm_segment column not available.")

## 5. Segment Revenue Contribution

In [ ]:
if 'rfm_segment' in customers.columns and 'monetary' in customers.columns:
    segment_revenue = customers.groupby('rfm_segment').agg(
        customers=('customer_id', 'count'),
        total_revenue=('monetary', 'sum'),
        avg_revenue=('monetary', 'mean')
    ).sort_values('total_revenue', ascending=False)
    
    segment_revenue['revenue_pct'] = (segment_revenue['total_revenue'] / segment_revenue['total_revenue'].sum() * 100).round(2)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    segment_revenue['total_revenue'].plot(kind='bar', ax=axes[0], color='seagreen')
    axes[0].set_title('Revenue by RFM Segment')
    axes[0].set_ylabel('Total Revenue')
    axes[0].tick_params(axis='x', rotation=45)
    
    segment_revenue['avg_revenue'].plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Avg Revenue per Customer by Segment')
    axes[1].set_ylabel('Avg Revenue')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print('\nSegment Revenue Summary:')
    print(segment_revenue.to_string())

## 6. RFM Heatmap

In [ ]:
if all(c in customers.columns for c in ['recency', 'frequency', 'monetary']) and 'rfm_segment' in customers.columns:
    customers['r_bin'] = pd.qcut(customers['recency'], q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], duplicates='drop')
    customers['f_bin'] = pd.qcut(customers['frequency'].rank(method='first'), q=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'], duplicates='drop')
    
    heatmap_data = customers.pivot_table(
        values='monetary',
        index='r_bin',
        columns='f_bin',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax)
    ax.set_title('Avg Monetary Value: Recency vs Frequency')
    ax.set_xlabel('Frequency')
    ax.set_ylabel('Recency')
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for heatmap not available.")

## 7. RFM Metrics Report

In [ ]:
metrics_path = Path('../outputs/metrics')

rfm_summary_path = metrics_path / 'rfm_summary.json'
if rfm_summary_path.exists():
    with open(rfm_summary_path, "r") as f:
        rfm_summary = json.load(f)
    print("RFM SUMMARY")
    print("=" * 60)
    print(json.dumps(rfm_summary, indent=2, default=str))
else:
    print("rfm_summary.json not found.")

In [ ]:
rfm_details_path = metrics_path / 'rfm_segment_details.json'
if rfm_details_path.exists():
    with open(rfm_details_path, "r") as f:
        rfm_details = json.load(f)
    print("\nRFM SEGMENT DETAILS")
    print("=" * 60)
    print(json.dumps(rfm_details, indent=2, default=str))
else:
    print("rfm_segment_details.json not found.")

## 8. Summary

In [ ]:
print('RFM Segmentation Summary')
print("=" * 60)
if "rfm_segment" in customers.columns:
    print(f"Total customers analyzed: {len(customers):,}")
    print(f"Number of segments: {customers['rfm_segment'].nunique()}")
    print(f"Segments: {list(customers['rfm_segment'].unique())}")
    print(f"\nTop segment by count: {customers['rfm_segment'].value_counts().idxmax()}")
    if "monetary" in customers.columns:
        print(f"Top segment by revenue: {customers.groupby('rfm_segment')['monetary'].sum().idxmax()}")
else:
    print("RFM segmentation not available.")